# Re-evaluate Existing Model

Score a logged model against a labeled eval dataset and upsert the result on the trial run. This uses the same `evaluate(...)` path as trial-time evaluation: per-label reports and predictions are written through the trial artifact seam.


In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
from IPython.display import display

import automl
from automl import data, eval, experiment, trial
from automl.eval import Auc, EvalSpec, LogLoss, Metric, ThresholdSweep

DRY_RUN = True


/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/.venv/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [3]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)

leaderboard = experiment.leaderboard(training_origin="all", n=20, session=active)
rows = list(leaderboard.rows)
if not rows:
    raise RuntimeError("run notebook 2 with RUN_AGENT=True or notebook 3 with RUN_TRIAL=True first")

deployed = rows[0]
deployed.to_dict()


{'project': 'example_homecredit',
 'repo_root': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor',
 'project_dir': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

{'schema_version': 1,
 'run_id': 'b29ea9099bc54a2582b84dfeb550513d',
 'slug': 'example_notebook_2_elasticnet',
 'strategy': 'baseline',
 'status': 'FINISHED',
 'primary_metric_name': 'eval.test.auc',
 'primary_metric_value': 0.631578947368421,
 'started_at': '2026-06-02T07:04:13.715000Z',
 'ended_at': '2026-06-02T07:04:53.140000Z',
 'parent_run_id': None,
 'dataset_hash': 'sha256:8471c462af97cf485e43b918cf671bcf44bccde57256d40d344e7ef4cafba91a',
 'trial_number': 1,
 'hypothesis': 'An elastic-net logistic regression applied to the WOE-encoded feature pool will establish a regularized linear baseline that is interpretable and well-suited to sparse credit risk features, providing a reliable AUC reference for the example_homecredit project.',
 'training_origin': 'automl',
 'training_time_s': 44.01695312501397,
 'n_features': None}

In [4]:
loaded = data.materialize(session=active)
run_config = active.config.require_run_config()
train = data.load_dataset(split_name=run_config.train_split, session=active)
holdout = data.load_dataset(split_name=run_config.eval_split, session=active)
target_col = loaded.dataset.target_column
hash_key = tuple(loaded.dataset.hash_key)

{
    "dataset_id": loaded.dataset.id,
    "target_column": target_col,
    "hash_key": hash_key,
}


{'dataset_id': 'v1_8471c462',
 'target_column': 'target',
 'hash_key': ('sk_id_curr',)}

In [5]:
eval_dataset, cached = eval.prepare_eval_dataset(
    session=active,
    dataset_id=loaded.dataset.id,
    split=active.config.require_run_config().eval_split,
)

result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    label="notebook_eval",
    overwrite=False,
)

{
    "label": result.label,
    "eval_dataset_id": result.eval_dataset_id,
    "metrics": result.metrics,
    "cached_dataset": cached,
    "cached_result": result.cached,
    "predictions_uri": result.predictions_uri,
}


2026/06/02 00:09:21 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['SPLITID', 'sk_id_curr', 'target']`. These inputs will be ignored.


{'label': 'notebook_eval',
 'eval_dataset_id': 'ev_9b3d9b70a8ad',
 'metrics': ({'name': 'auc',
   'value': 0.631578947368421,
   'augmentations': []},),
 'cached_dataset': True,
 'cached_result': False,
 'predictions_uri': 'gs://automl-homecredit-kaggle-wliu/automl/dry_run/example_homecredit/example-homecredit/runs/2026-06/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_eval/predictions.parquet'}

In [6]:
metrics_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(
        primary=Auc(),
        metrics=[
            -LogLoss(),
            ThresholdSweep(thresholds=[0.3, 0.5, 0.7]),
        ],
    ),
    label="notebook_eval_custom_metrics",
    overwrite=False,
)

{
    "label": metrics_result.label,
    "primary": metrics_result.primary,
    "scalar_metrics": metrics_result.metrics,
    "cached_result": metrics_result.cached,
}


2026/06/02 00:09:27 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['SPLITID', 'sk_id_curr', 'target']`. These inputs will be ignored.


{'label': 'notebook_eval_custom_metrics',
 'primary': 'auc',
 'scalar_metrics': ({'name': 'auc',
   'value': 0.631578947368421,
   'augmentations': []},
  {'name': 'negative_log_loss',
   'value': -0.6185507328218643,
   'augmentations': []},
  {'name': 'threshold_sweep',
   'value': [{'threshold': 0.3,
     'precision': 0.14285714285714285,
     'recall': 0.5},
    {'threshold': 0.5, 'precision': 0.2, 'recall': 0.5},
    {'threshold': 0.7, 'precision': 0.3333333333333333, 'recall': 0.5}],
   'augmentations': []}),
 'cached_result': False}

In [7]:
primary_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(
        primary=-LogLoss(),
        metrics=[
            Auc(),
            ThresholdSweep(thresholds=[0.3, 0.5, 0.7]),
        ],
    ),
    label="notebook_eval_primary_logloss",
    set_as_primary_label=True,
    overwrite=False,
)

{
    "label": primary_result.label,
    "primary": primary_result.primary,
    "primary_label": primary_result.label,
    "cached_pointer_update": primary_result.cached,
}


2026/06/02 00:09:34 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['SPLITID', 'sk_id_curr', 'target']`. These inputs will be ignored.


{'label': 'notebook_eval_primary_logloss',
 'primary': 'negative_log_loss',
 'primary_label': 'notebook_eval_primary_logloss',
 'cached_pointer_update': False}

In [8]:
class WeightedMeanScore(Metric):
    required_augmentations = ("risk_weight",)
    required_columns = ("RISK_WEIGHT",)

    def compute(self, df_test, y_pred, target_col):
        return float(np.average(y_pred, weights=df_test["RISK_WEIGHT"]))


risk_weight_frame = holdout.df.loc[:, list(hash_key)].copy()
risk_weight_frame["RISK_WEIGHT"] = np.linspace(1.0, 2.0, len(risk_weight_frame))
risk_weight, risk_weight_cached = eval.prepare_eval_augmentation(
    session=active,
    eval_dataset_id=eval_dataset.id,
    frame=risk_weight_frame,
    name="risk_weight",
)

augmented_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(primary=WeightedMeanScore(), metrics=[Auc()]),
    label="notebook_eval_with_risk_weight",
    overwrite=False,
)

{
    "augmentation": risk_weight.data_gcs_uri,
    "augmentation_cached": risk_weight_cached,
    "label": augmented_result.label,
    "primary": augmented_result.primary,
    "metrics": augmented_result.metrics,
    "cached_result": augmented_result.cached,
}


2026/06/02 00:09:50 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['SPLITID', 'sk_id_curr', 'target']`. These inputs will be ignored.


{'augmentation': 'gs://automl-homecredit-kaggle-wliu/automl/dry_run/example_homecredit/example-homecredit/eval/datasets/ev_9b3d9b70a8ad/augmentations/risk_weight__860dd40a/data.parquet',
 'augmentation_cached': False,
 'label': 'notebook_eval_with_risk_weight',
 'primary': 'weighted_mean_score',
 'metrics': ({'name': 'weighted_mean_score',
   'value': 0.31405782128368065,
   'augmentations': ['risk_weight']},
  {'name': 'auc', 'value': 0.631578947368421, 'augmentations': []}),
 'cached_result': False}

In [9]:
external_parts = [
    group.head(min(10, len(group)))
    for _, group in train.df.groupby(target_col, sort=True)
]
external_frame = pd.concat(external_parts, ignore_index=True)
if external_frame[target_col].nunique() < 2:
    raise RuntimeError("external eval sample must contain both target classes for AUC")

external_eval, external_cached = eval.prepare_eval_dataset(
    session=active,
    kind="external",
    frame=external_frame,
    target_col=target_col,
    hash_key=hash_key,
    provenance={"source": "notebook_5_external_labeled_sample"},
)

external_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=external_eval.id,
    eval_spec=EvalSpec(primary=Auc(), metrics=[-LogLoss()]),
    label="notebook_external_labeled_sample",
    overwrite=False,
)

{
    "eval_dataset_id": external_result.eval_dataset_id,
    "kind": external_eval.kind,
    "cached_dataset": external_cached,
    "label": external_result.label,
    "metrics": external_result.metrics,
    "predictions_uri": external_result.predictions_uri,
}


2026/06/02 00:09:59 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['SPLITID', 'sk_id_curr', 'target']`. These inputs will be ignored.


{'eval_dataset_id': 'ev_fafd310c9a94',
 'kind': 'external',
 'cached_dataset': False,
 'label': 'notebook_external_labeled_sample',
 'metrics': ({'name': 'auc', 'value': 1.0, 'augmentations': []},
  {'name': 'negative_log_loss',
   'value': -0.22768858258884045,
   'augmentations': []}),
 'predictions_uri': 'gs://automl-homecredit-kaggle-wliu/automl/dry_run/example_homecredit/example-homecredit/runs/2026-06/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_external_labeled_sample/predictions.parquet'}

In [10]:
details = trial.show_trial(deployed.run_id, session=active)
details.evaluations


(EvalResult(label='notebook_eval', eval_dataset_id='ev_9b3d9b70a8ad', eval_dataset_kind='split_view', predictions_uri='gs://automl-homecredit-kaggle-wliu/automl/dry_run/example_homecredit/example-homecredit/runs/2026-06/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_eval/predictions.parquet', predictions_manifest_uri='runs:/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_eval/predictions.json', augmentations_used=(), primary='auc', metrics=({'name': 'auc', 'value': 0.631578947368421, 'augmentations': []},), computed_at='2026-06-02T07:09:21.734788+00:00', schema_version=1, cached=False),
 EvalResult(label='notebook_eval_custom_metrics', eval_dataset_id='ev_9b3d9b70a8ad', eval_dataset_kind='split_view', predictions_uri='gs://automl-homecredit-kaggle-wliu/automl/dry_run/example_homecredit/example-homecredit/runs/2026-06/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_eval_custom_metrics/predictions.parquet', predictions_manifest_uri='runs:/b29ea9099bc54a2582b84dfeb550513d/eval/notebook_eval